In [1]:
import sys
sys.path.insert(0, "..")

import joblib

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from loguru import logger

pd.set_option("display.max_columns", None)

In [2]:
from src.evaluation.metrics import build_predictions_report

In [3]:
def plot_barh_by_model(results_df, x, y="model", hue=None, title="", figsize=(8, 6)):
    """Barplot horizontal genérico para comparar modelos por una métrica."""
    df_sorted = results_df.sort_values(x, ascending=True)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(df_sorted, x=x, y=y, hue=hue, order=df_sorted[y], ax=ax)
    ax.set_title(title, loc="left", pad=20)
    ax.set_xlabel("")
    ax.set_ylabel("")
    sns.despine()
    if hue:
        ax.legend(title=hue, loc="upper right")
    plt.tight_layout()
    plt.show()

In [4]:
!ls ../data/datasets/daily/

dataset_level_01_daily_total.parquet
dataset_level_02_daily_state.parquet
dataset_level_03_daily_cat.parquet
dataset_level_04_daily_dept.parquet
dataset_level_05_daily_state_cat.parquet
dataset_level_06_daily_store.parquet
dataset_level_07_daily_state_dept.parquet
dataset_level_08_daily_store_cat.parquet
dataset_level_09_daily_store_dept.parquet
dataset_level_10_daily_item__FOODS_1.parquet
dataset_level_10_daily_item__FOODS_2.parquet
dataset_level_10_daily_item__FOODS_3.parquet
dataset_level_10_daily_item__HOBBIES_1.parquet
dataset_level_10_daily_item__HOBBIES_2.parquet
dataset_level_10_daily_item__HOUSEHOLD_1.parquet
dataset_level_10_daily_item__HOUSEHOLD_2.parquet
dataset_level_11_daily_item_state__CA_FOODS_1.parquet
dataset_level_11_daily_item_state__CA_FOODS_2.parquet
dataset_level_11_daily_item_state__CA_FOODS_3.parquet
dataset_level_11_daily_item_state__CA_HOBBIES_1.parquet
dataset_level_11_daily_item_state__CA_HOBBIES_2.parquet
dataset_level_11_daily_item_state__CA_HOUSEHOLD_1.p

In [ ]:
LEVEL = 'level_01_daily_total'
TARGET = 'sales'

In [ ]:
import joblib

artifact_path = f"../artifacts/models/{LEVEL}_{TARGET}_artifact.pkl"
artifact = joblib.load(artifact_path)

model = artifact["model"]
X_test, y_test = artifact["X_test"], artifact["y_test"]
FEATURES = artifact["features"]
feature_importance = artifact["feature_importance"]
results_df = artifact["results_df"]

In [ ]:
#['wape', 'wrmsse', 'mae', 'rmse', 'mape', 'smape', 'bias', 'rmsle', 'tracking_signal', 'spec', 'mase', 'fit_time']:
for col in ['wape', 'wrmsse', 'spec']:
    plot_barh_by_model(results_df, hue="category", x=col, y="model", title=f"{col} por modelo")

## Explicación modelo

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

X_shap = X_test.sample(n=min(30_000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)
shap_df = pd.DataFrame(shap_values, columns=X_shap.columns, index=X_shap.index)

importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

In [ ]:
shap.summary_plot(shap_values, X_shap, plot_type="bar")

In [ ]:
shap.summary_plot(shap_values, X_shap)

# Predicciones

In [ ]:
from src.evaluation import analizar_prediccion as _analizar_prediccion

def analizar_prediccion(agg_id, test=artifact['test'], target=artifact['target'],
                         explainer=explainer, date=None, max_display=10):
    return _analizar_prediccion(
        test, X_test, df_pred, target, agg_id,
        date=date, max_display=max_display, explainer=explainer,
    )

In [ ]:
metrics_test_final, df_pred = build_predictions_report(artifact['train'], artifact['test'], y_test, model.predict(X_test), target_col=artifact['target'])

print(f"Test WAPE: {metrics_test_final['wape']:.2%}")
print(f"Test WRMSSE: {metrics_test_final['wrmsse']:.4f}")

In [ ]:
from src.evaluation.metrics import build_series_metrics

df_metrics = build_series_metrics(artifact['train'], df_pred, target_col=artifact['target'], weight_level=["date"])
df_metrics.head(10)

In [ ]:
df_metrics.head(10)

In [ ]:
df_pred.query('agg_id == "FOODS_1_FOODS_FOODS_1_110_CA_1_CA"')

In [ ]:
analizar_prediccion(agg_id='FOODS_1_FOODS_FOODS_1_110_CA_1_CA',date='2016-05-18')